# GenLab — Launcher
Plataforma modular para ejecutar modelos open source de IA en Colab.

**Antes de ejecutar:**
1. Runtime → Cambiar tipo de entorno de ejecución → **T4 GPU**.
2. Consigue tu **HF_TOKEN** (ver celda 5).
3. Concede permisos de Drive cuando se solicite.

**IMPORTANTE:** Ejecuta las celdas EN ORDEN, de arriba hacia abajo. Nunca saltes celdas.

In [ ]:
# 1. Instalar dependencias
!pip install -q torch "diffusers>=0.33.0" transformers huggingface-hub imageio imageio-ffmpeg accelerate psutil pyyaml hf_transfer

In [ ]:
# 2. Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clonar repo, cargar genlab y bootstrap
import os, sys

REPO_URL = 'https://github.com/ke1npro/GenLab.git'
GENLAB_SRC = '/content/genlab'

# Clonar o actualizar
if os.path.isdir(GENLAB_SRC):
    !git -C $GENLAB_SRC pull
else:
    !git clone $REPO_URL $GENLAB_SRC

# Verificar que el clone funcionó
assert os.path.isdir(f'{GENLAB_SRC}/src/genlab'), \
    f'No se encontró src/genlab/ en {GENLAB_SRC} — el clone falló'

# Agregar src/ al path de Python y cambiar al repo
sys.path.insert(0, f'{GENLAB_SRC}/src')
os.chdir(GENLAB_SRC)

# Importar y bootstrap
from genlab import GenLab
print('genlab importado OK')

info = GenLab.bootstrap()

# Checklist visual
hw = info['hardware']
print()
print('--- Checklist ---')
print(f'[{"OK" if hw["has_gpu"] else "XX"}] GPU: {hw["gpu"]}')
print(f'[{"OK" if info["hf_token_ok"] else "XX"}] HF_TOKEN: {"configurado" if info["hf_token_ok"] else "pendiente"}')
print(f'[{"OK" if info.get("hf_transfer") else "--"}] hf_transfer: {"Disponible" if info.get("hf_transfer") else "No instalado"}')
print(f'[{"OK"}]          genlab importado')

In [ ]:
# 4. Token de Hugging Face
# ¿Dónde conseguirlo?
# 1. https://huggingface.co/join — crear cuenta
# 2. https://huggingface.co/settings/tokens — New token → Role: Read
# 3. Copia el token (empieza con 'hf_')
# 4. https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers — (no requiere gated, Apache 2.0)

try:
    from genlab import GenLab
except ModuleNotFoundError:
    raise RuntimeError('genlab no está cargado. Ejecuta primero la CELDA 3 (la anterior). Si ya la ejecutaste, haz Runtime → Restart and run all para empezar fresco.')

from google.colab import userdata
import os
try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('[OK] HF_TOKEN cargado desde Secrets de Colab.')
except Exception:
    token = input('Pega tu HF_TOKEN (o Enter para omitir): ').strip()
    if token:
        os.environ['HF_TOKEN'] = token
        print('[OK] HF_TOKEN configurado manualmente.')
    else:
        print('[XX] HF_TOKEN no configurado.')

In [ ]:
# 5. Elegir modelo
# Selecciona el modelo base:
#   wan       → Wan2.1 T2V 1.3B (recomendado, ~9 GB descarga)
#   cogvideo  → CogVideoX-2b (~9 GB descarga, más ligero)
#
# También puedes especificar un model_id personalizado (mirror, variante cuantizada, etc.)

MODELOS_DISPONIBLES = {
    "wan": {
        "label": "Wan2.1 T2V 1.3B (recomendado)",
        "default_model_id": "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    },
    "cogvideo": {
        "label": "CogVideoX-2b (~9 GB)",
        "default_model_id": "zai-org/CogVideoX-2b",
    },
}

# --- CONFIGURA AQUÍ ---
modelo_elegido = "wan"  # cambia a "cogvideo" si quieres el más ligero
model_id_personalizado = ""  # o pega un model_id custom (ej: "THUDM/CogVideoX-2b")
# -------------------------

if model_id_personalizado:
    modelo_final = modelo_elegido
    config_extra = {'model': {'model_id': model_id_personalizado}}
    print(f'[OK] Usando model_id personalizado: {model_id_personalizado}')
else:
    modelo_final = modelo_elegido
    info_m = MODELOS_DISPONIBLES[modelo_elegido]
    config_extra = {}
    print(f'[OK] Modelo: {info_m["label"]} ({info_m["default_model_id"]})')

In [ ]:
# 6. Generar video
# Descarga el modelo (~5-9 GB primera vez), carga en GPU y genera.
# En T4 puede tomar 4-6 minutos.

try:
    from genlab import GenLab
except ModuleNotFoundError:
    raise RuntimeError('genlab no está cargado. Ejecuta primero la CELDA 3.')

print(f'[..] Iniciando descarga de {modelo_final}...')

result = GenLab().run(
    task='text_to_video',
    model=modelo_final,
    prompt='Un astronauta montando un caballo en la luna, estilo cinematográfico',
    config=config_extra,
)

print(f'[OK] Video generado: {result["video_path"]}')

In [ ]:
# 7. Mostrar resultado
from IPython.display import Video
Video(result['video_path'], width=480)